In [91]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.utilities import SQLDatabase
from langchain.embeddings import OllamaEmbeddings
from langchain.llms import Ollama
from langchain.vectorstores import Chroma
from langchain.prompts import SemanticSimilarityExampleSelector
from langchain.prompts.prompt import PromptTemplate
from langchain.chains.sql_database.prompt import PROMPT_SUFFIX, _mysql_prompt 
from langchain.prompts import FewShotPromptTemplate
from langchain_community.agent_toolkits import create_sql_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents.agent_types import AgentType

In [ ]:
MODEL_API_KEY = "################"

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.0-flash-exp",
    google_api_key = MODEL_API_KEY
)

In [93]:
db_user = 'root'
db_pass = ''
db_host = 'localhost'
db_name = 'human_resouce'

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_pass}@{db_host}/{db_name}", sample_rows_in_table_info = 3)
print(db.table_info)


CREATE TABLE countries (
	country_id CHAR(2) NOT NULL, 
	country_name VARCHAR(40), 
	region_id INTEGER(11) NOT NULL, 
	PRIMARY KEY (country_id), 
	CONSTRAINT countries_ibfk_1 FOREIGN KEY(region_id) REFERENCES regions (region_id) ON DELETE CASCADE ON UPDATE CASCADE
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB

/*
3 rows from countries table:
country_id	country_name	region_id
BR	Brazil	2
DE	Germany	1
JP	Japan	3
*/


CREATE TABLE departments (
	department_id INTEGER(11) NOT NULL AUTO_INCREMENT, 
	department_name VARCHAR(30) NOT NULL, 
	location_id INTEGER(11), 
	PRIMARY KEY (department_id), 
	CONSTRAINT departments_ibfk_1 FOREIGN KEY(location_id) REFERENCES locations (location_id) ON DELETE CASCADE ON UPDATE CASCADE
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB

/*
3 rows from departments table:
department_id	department_name	location_id
1	Executive	1
2	IT	2
3	Marketing	3
*/


CREATE TABLE dependents (
	dependent_id INTEGER(11) NOT NULL AUTO_INCREMENT, 
	first_name VARCHAR(50) NOT NULL, 
	last_name VA

In [94]:
few_shorts = [
    {
        "Question":"List all employees' first and last names.",
        "SQLQuery":"SELECT first_name, last_name FROM employees;",
        "SQLResult":"""first_name, last_name
        
                       Jhon        Smith
                       Sarah       Johnson
                       Michael     Williams 
                       Emily       Brown
                       David       Jones
                       Jennifer    Garcia
                       Robert      Miller
                       Lisa        Davis
                       Thomas      Rodriguez
                       Patricia    Martinez

                    """,
        "Answer": """first_name, last_name
        
                       Jhon        Smith
                       Sarah       Johnson
                       Michael     Williams 
                       Emily       Brown
                       David       Jones
                       Jennifer    Garcia
                       Robert      Miller
                       Lisa        Davis
                       Thomas      Rodriguez
                       Patricia    Martinez

                """
    },
    
    {
        "Question":"Show all employees hired after 2015",
        "SQLQuery":"SELECT first_name, last_name, hire_date FROM employees WHERE hire_date > '2015-01-01';",
        "SQLResult":"""first_name  last_name  hire_date
        
                       Jennifer    Garcia     2015-02-18
                       Robert      Miller     2016-04-30
                       Lisa        Davis      2017-08-12
                       Thomas      Rodriguez  2018-01-25
                       Patricia    Martinez   2019-05-08
                       

                    """,
        "Answer": """first_name  last_name    hire_date
        
                       Jennifer    Garcia     2015-02-18
                       Robert      Miller     2016-04-30
                       Lisa        Davis      2017-08-12
                       Thomas      Rodriguez  2018-01-25
                       Patricia    Martinez   2019-05-08
                       

                    """
    },
    
    
    {
        "Question":"Display the first 3 employees in the database",
        "SQLQuery":"SELECT * FROM employees LIMIT 3;",
        "SQLResult":""" employee_id  first_name  last_name          email                     phone_number      hire_date       job_id     salary       manager_id      department_id
                            
                            1           John        Smith      john.smith@company.com          555-1001         2010-06-01        1        25000.00      NULL               1
                            2           Sarah       Johnson    sarah.johnson@company.com       555-1002         2011-07-15        2        18000.00       1                 1
                            3           Michael     Williams   michael.williams@company.com    555-1003         2012-03-10        3        8000.00        2                 2

                       

                    """,
        "Answer": """ employee_id  first_name  last_name          email                     phone_number      hire_date       job_id     salary       manager_id      department_id
                            
                            1           John        Smith      john.smith@company.com          555-1001         2010-06-01        1        25000.00      NULL               1
                            2           Sarah       Johnson    sarah.johnson@company.com       555-1002         2011-07-15        2        18000.00       1                 1
                            3           Michael     Williams   michael.williams@company.com    555-1003         2012-03-10        3        8000.00        2                 2

                       

                    """
    },
    
    
    {
        "Question":"How many employees are there?",
        "SQLQuery":"SELECT COUNT(*) FROM employees;",
        "SQLResult":"[(10,)]",
        "Answer": "10"
    },
    
    
    {
        "Question":"What is the total salary of all employees?",
        "SQLQuery":"SELECT SUM(salary) FROM employees;",
        "SQLResult":"[(94500.00,)]",
        "Answer": "94500.00"
    },
    
    
    {
        "Question":"What is the average salary",
        "SQLQuery":"SELECT AVG(salary) FROM employees;",
        "SQLResult":"[(9450.000000,)]",
        "Answer": "9450.000000"
    },
    
    
    {
        "Question":"Show the number of employees in each department.",
        "SQLQuery":"SELECT department_id, COUNT(*) FROM employees GROUP BY department_id;",
        "SQLResult":"""department_id      COUNT(*)
                             
                             1               2
                             2               3
                             3               1
                             4               4
                    """,
        "Answer": """department_id      COUNT(*)
                             
                             1               2
                             2               3
                             3               1
                             4               4
                    """
    },
    
    
    
    {
        "Question":"List departments with more than 2 employees",
        "SQLQuery":"SELECT department_id, COUNT(*) AS emp_count FROM employees GROUP BY department_id HAVING emp_count > 2;",
        "SQLResult":"""department_id      emp_count
                             
                             2               3
                             4               4
                    """,
        "Answer": """department_id      emp_count
                             
                             2               3
                             4               4
                    """
    },
    
    
    
    {
        "Question":"List employees ordered by salary ascending",
        "SQLQuery":"SELECT first_name, last_name, salary FROM employees ORDER BY salary ASC;",
        "SQLResult":"""first_name      last_name      salary
                        
                        Patricia        Martinez      3500.00 
                        Thomas          Rodriguez     4000.00
                        Lisa            Davis         4500.00
                        David           Jones         5000.00
                        Robert          Miller        7000.00
                        Jennifer        Garcia        7500.00
                        Michael         Williams      8000.00
                        Emily           Brown         12000.00
                        Sarah           Johnson       18000.00
                        John            Smith         25000.00             
                    """,
        "Answer": """first_name      last_name      salary
                        
                        Patricia        Martinez      3500.00 
                        Thomas          Rodriguez     4000.00
                        Lisa            Davis         4500.00
                        David           Jones         5000.00
                        Robert          Miller        7000.00
                        Jennifer        Garcia        7500.00
                        Michael         Williams      8000.00
                        Emily           Brown         12000.00
                        Sarah           Johnson       18000.00
                        John            Smith         25000.00             
                    """
    },
    
    
    
    
    {
        "Question":"List top 5 highest-paid employees.",
        "SQLQuery":"SELECT first_name, last_name, salary FROM employees ORDER BY salary DESC LIMIT 5;",
        "SQLResult":"""first_name      last_name      salary
                             
                         John           Smith         25000.00
                         Sarah          Johnson       18000.00
                         Emily          Brown         12000.00
                         Michael        Williams      8000.00
                         Jennifer       Garcia        7500.00
                              
                    """,
        "Answer": """first_name      last_name      salary
                             
                         John           Smith         25000.00
                         Sarah          Johnson       18000.00
                         Emily          Brown         12000.00
                         Michael        Williams      8000.00
                         Jennifer       Garcia        7500.00
                              
                    """
    },
    
    
    
    
    {
        "Question":"List employees with their department names.",
        "SQLQuery":"SELECT e.first_name, e.last_name, d.department_name FROM employees e JOIN departments d ON e.department_id = d.department_id;",
        "SQLResult":"""first_name    last_name    department_name
                             
                         John         Smith        Executive
                         Sarah        Johnson      Executive
                         Michael      Williams     IT
                         Jennifer     Garcia       IT
                         Robert       Miller       IT
                         Emily        Brown        Marketing
                         David        Jones        Sales
                         Lisa         Davis        Sales
                         Thomas       Rodriguez    Sales
                         Patricia     Martinez     Sales
                              
                    """,
        "Answer": """first_name    last_name    department_name
                             
                         John         Smith        Executive
                         Sarah        Johnson      Executive
                         Michael      Williams     IT
                         Jennifer     Garcia       IT
                         Robert       Miller       IT
                         Emily        Brown        Marketing
                         David        Jones        Sales
                         Lisa         Davis        Sales
                         Thomas       Rodriguez    Sales
                         Patricia     Martinez     Sales
                              
                    """
    },
    
    
    
     {
        "Question":"List employees with their job titles",
        "SQLQuery":"SELECT e.first_name, e.last_name, j.job_title FROM employees e JOIN jobs j ON e.job_id = j.job_id;",
        "SQLResult":"""first_name  last_name  job_title
                                     
                         John       Smith      President
                         Sarah      Johnson    Administration Vice President
                         Michael    Williams   Programmer
                         Jennifer   Garcia     Programmer
                         Robert     Miller     Programmer
                         Emily      Brown      Marketing Manager
                         David      Jones      Sales Representative
                         Lisa       Davis      Sales Representative
                         Thomas     Rodriguez  Sales Representative
                         Patricia   Martinez   Sales Representative
                              
                    """,
        "Answer": """first_name  last_name  job_title
                                     
                         John       Smith      President
                         Sarah      Johnson    Administration Vice President
                         Michael    Williams   Programmer
                         Jennifer   Garcia     Programmer
                         Robert     Miller     Programmer
                         Emily      Brown      Marketing Manager
                         David      Jones      Sales Representative
                         Lisa       Davis      Sales Representative
                         Thomas     Rodriguez  Sales Representative
                         Patricia   Martinez   Sales Representative
                              
                    """
    },
    
    
    
    {
        "Question":"Show employee names, department names, and city",
        "SQLQuery":"SELECT e.first_name, e.last_name, d.department_name, l.city FROM employees e JOIN departments d ON e.department_id = d.department_id JOIN locations l ON d.location_id = l.location_id;",
        "SQLResult":"""first_name       last_name      department_name        city
        
                         John             Smith           Executive          New York
                         Sarah            Johnson         Executive          New York
                         Michael          Williams        IT                 Berlin
                         Jennifer         Garcia          IT                 Berlin
                         Robert           Miller          IT                 Berlin
                         Emily            Brown           Marketing          Tokyo
                         David            Jones           Sales              Lagos
                         Lisa             Davis           Sales              Lagos
                         Thomas           Rodriguez       Sales              Lagos
                         Patricia         Martinez        Sales              Lagos
                              
                    """,
        "Answer": """first_name       last_name      department_name        city
        
                         John             Smith           Executive          New York
                         Sarah            Johnson         Executive          New York
                         Michael          Williams        IT                 Berlin
                         Jennifer         Garcia          IT                 Berlin
                         Robert           Miller          IT                 Berlin
                         Emily            Brown           Marketing          Tokyo
                         David            Jones           Sales              Lagos
                         Lisa             Davis           Sales              Lagos
                         Thomas           Rodriguez       Sales              Lagos
                         Patricia         Martinez        Sales              Lagos
                              
                    """
    },
    
    
    {
        "Question":"Find employees in the 'IT' department.",
        "SQLQuery":"SELECT e.first_name, e.last_name FROM employees e JOIN departments d ON e.department_id = d.department_id WHERE d.department_name = 'IT';",
        "SQLResult":"""first_name       last_name   
        
                        Michael         Williams
                        Jennifer        Garcia
                        Robert          Miller
                              
                    """,
        "Answer": """first_name       last_name   
        
                        Michael         Williams
                        Jennifer        Garcia
                        Robert          Miller
                              
                    """
    },
    
    
    
    {
        "Question":"Show employee and region name.",
        "SQLQuery":"SELECT e.first_name, e.last_name, r.region_name FROM employees e JOIN departments d ON e.department_id = d.department_id JOIN locations l ON d.location_id = l.location_id JOIN countries c ON l.country_id = c.country_id JOIN regions r ON c.region_id = r.region_id;",
        "SQLResult":"""first_name      last_name      region_name   
        
                        Michael        Williams          Europe
                        Jennifer       Garcia            Europe
                        Robert         Miller            Europe
                        John           Smith             Americas
                        Sarah          Johnson           Americas
                        Emily          Brown             Asia
                        David          Jones             Middle East and Africa
                        Lisa           Davis             Middle East and Africa
                        Thomas         Rodriguez         Middle East and Africa
                        Patricia       Martinez          Middle East and Africa
                              
                    """,
        "Answer": """first_name      last_name      region_name   
        
                        Michael        Williams          Europe
                        Jennifer       Garcia            Europe
                        Robert         Miller            Europe
                        John           Smith             Americas
                        Sarah          Johnson           Americas
                        Emily          Brown             Asia
                        David          Jones             Middle East and Africa
                        Lisa           Davis             Middle East and Africa
                        Thomas         Rodriguez         Middle East and Africa
                        Patricia       Martinez          Middle East and Africa
                              
                    """
    },
    
    
    
    {
        "Question":"Employees earning above average salary.",
        "SQLQuery":"SELECT first_name, last_name, salary FROM employees WHERE salary > (SELECT AVG(salary) FROM employees);",
        "SQLResult":"""first_name      last_name      salary   
        
                        John            Smith         25000.00
                        Sarah           Johnson       18000.00
                        Emily           Brown         12000.00
                              
                    """,
        "Answer": """first_name      last_name      salary   
        
                        John            Smith         25000.00
                        Sarah           Johnson       18000.00
                        Emily           Brown         12000.00
                              
                    """
    },
    
    
    {
        "Question":"Employees in departments located in Japan",
        "SQLQuery":"SELECT e.first_name, e.last_name FROM employees e WHERE e.department_id IN (SELECT d.department_id FROM departments d JOIN locations l ON d.location_id = l.location_id WHERE l.country_id = 'JP');",
        "SQLResult":"""first_name      last_name        

                        Emily           Brown   
                              
                    """,
        "Answer": """first_name      last_name        

                        Emily           Brown   
                              
                    """
    },
    
    
    
    {
        "Question":"Employees without dependents.",
        "SQLQuery":"SELECT first_name, last_name FROM employees WHERE employee_id NOT IN (SELECT employee_id FROM dependents);",
        "SQLResult":"""first_name      last_name        

                        Patricia        Martinez   
                              
                    """,
        "Answer": """first_name      last_name        

                      Patricia        Martinez   
                              
                    """
    },
    
    
    {
        "Question":"Employees managed by 'Sarah Johnson'",
        "SQLQuery":"SELECT COUNT(*) FROM employees WHERE manager_id = (SELECT employee_id FROM employees WHERE first_name = 'Sarah' AND last_name = 'Johnson');",
        "SQLResult":"[(2,)]",
        "Answer": "2"
    },
    
    
    
    
    {
        "Question":"List dependents with their parent employees.",
        "SQLQuery":"SELECT d.first_name AS dep_first, d.last_name AS dep_last, e.first_name AS emp_first, e.last_name AS emp_last FROM dependents d JOIN employees e ON d.employee_id = e.employee_id;",
        "SQLResult":"""dep_first     dep_last     emp_first     emp_last        

                        Anna           Smith        John          Smith
                        Benjamin       Jones        David         Jones
                        Charlotte      Davis        Lisa          Davis
                        Henry          Rodriguez    Thomas        Rodriguez
                        James          Smith        John          Smith
                        Lucas          Miller       Robert        Miller
                        Mia            Garcia       Jennifer      Garcia
                        Olivia         Johnson      Sarah         Johnson
                        Sophia         Brown        Emily         Brown
                        William        Williams     Michael       Williams   
                              
                    """,
        "Answer": """dep_first     dep_last     emp_first     emp_last        

                        Anna           Smith        John          Smith
                        Benjamin       Jones        David         Jones
                        Charlotte      Davis        Lisa          Davis
                        Henry          Rodriguez    Thomas        Rodriguez
                        James          Smith        John          Smith
                        Lucas          Miller       Robert        Miller
                        Mia            Garcia       Jennifer      Garcia
                        Olivia         Johnson      Sarah         Johnson
                        Sophia         Brown        Emily         Brown
                        William        Williams     Michael       Williams   
                              
                    """,
    },
    
    
    
    
    {
        "Question":"List Employees and their managers.",
        "SQLQuery":"SELECT e.first_name AS employee, m.first_name AS manager FROM employees e LEFT JOIN employees m ON e.manager_id = m.employee_id;",
        "SQLResult":"""employee       manager        
                       
                        David          Emily
                        Emily          Sarah
                        Jennifer       Michael
                        John 
                        Lisa           David
                        Michael        Sarah
                        Patricia       David
                        Robert         Michael
                        Sarah          John
                        Thomas         David
    
                              
                    """,
        "Answer": """employee       manager        
                      
                        David          Emily
                        Emily          Sarah
                        Jennifer       Michael
                        John 
                        Lisa           David
                        Michael        Sarah
                        Patricia       David
                        Robert         Michael
                        Sarah          John
                        Thomas         David
    
                              
                    """
    },
    
    
    
    {
        "Question":"Show number of dependents per employee",
        "SQLQuery":"SELECT e.first_name, COUNT(d.dependent_id) FROM employees e LEFT JOIN dependents d ON e.employee_id = d.employee_id GROUP BY e.employee_id;",
        "SQLResult":"""first_name   COUNT(d.dependent_id)        
                       
                        John                  2
                        Sarah                 1
                        Michael               1
                        Emily                 1
                        David                 1
                        Jennifer              1
                        Robert                1
                        Lisa                  1
                        Thomas                1
                        Patricia              0
    
                              
                    """,
        "Answer": """first_name   COUNT(d.dependent_id)        
                       
                        John                  2
                        Sarah                 1
                        Michael               1
                        Emily                 1
                        David                 1
                        Jennifer              1
                        Robert                1
                        Lisa                  1
                        Thomas                1
                        Patricia              0
    
                              
                    """,
    },
    
    
    {
        "Question":"Employees with email ending in '@company.com'",
        "SQLQuery":"SELECT first_name, last_name, email FROM employees WHERE email LIKE '%@company.com';",
        "SQLResult":"""first_name      last_name,       email        
                       
                        John            Smith           john.smith@company.com
                        Sarah           Johnson         sarah.johnson@company.com
                        Michael         Williams        michael.williams@company.com
                        Emily           Brown           emily.brown@company.com
                        David           Jones           david.jones@company.com
                        Jennifer        Garcia          jennifer.garcia@company.com
                        Robert          Miller          robert.miller@company.com
                        Lisa            Davis           lisa.davis@company.com
                        Thomas          Rodriguez       thomas.rodriguez@company.com
                        Patricia        Martinez        patricia.martinez@company.com
    
                              
                    """,
        "Answer": """first_name      last_name,       email        
                       
                        John            Smith           john.smith@company.com
                        Sarah           Johnson         sarah.johnson@company.com
                        Michael         Williams        michael.williams@company.com
                        Emily           Brown           emily.brown@company.com
                        David           Jones           david.jones@company.com
                        Jennifer        Garcia          jennifer.garcia@company.com
                        Robert          Miller          robert.miller@company.com
                        Lisa            Davis           lisa.davis@company.com
                        Thomas          Rodriguez       thomas.rodriguez@company.com
                        Patricia        Martinez        patricia.martinez@company.com
    
                              
                    """,
    },
    
    
    
    {
        "Question":"Employees with salary between 7000 and 10000.",
        "SQLQuery":"SELECT first_name, last_name, salary FROM employees WHERE salary BETWEEN 7000 AND 10000;",
        "SQLResult":"""first_name,      last_name,      salary        
                       
                        Michael         Williams        8000.00
                        Jennifer        Garcia          7500.00
                        Robert          Miller          7000.00
    
                              
                    """,
        "Answer": """first_name,      last_name,      salary        
                       
                        Michael         Williams        8000.00
                        Jennifer        Garcia          7500.00
                        Robert          Miller          7000.00
        
                    """
    },
    
    
    
    
    {
        "Question":"Which cities have departments?",
        "SQLQuery":"SELECT DISTINCT l.city FROM departments d JOIN locations l ON d.location_id = l.location_id;",
        "SQLResult":"""city 
        
                        Berlin
                        Lagos
                        New York
                        São Paulo
                        Tokyo
    
                              
                    """,
        "Answer": """city 
        
                        Berlin
                        Lagos
                        New York
                        São Paulo
                        Tokyo
    
                              
                    """
    },
    
    
    
    {
        "Question":"All departments and how many employees in each.",
        "SQLQuery":"SELECT d.department_name, COUNT(e.employee_id) FROM departments d LEFT JOIN employees e ON d.department_id = e.department_id GROUP BY d.department_name;",
        "SQLResult":"""department_name     COUNT(e.employee_id) 
        
                          Executive                 2
                          HR                        0 
                          IT                        3
                          Marketing                 1
                          Sales                     4
    
                              
                    """,
        "Answer": """department_name,    COUNT(e.employee_id) 
        
                          Executive                 2
                          HR                        0 
                          IT                        3
                          Marketing                 1
                          Sales                     4
    
                              
                    """
    },
    
    
    
    {
        "Question":"Departments with their location and city",
        "SQLQuery":"SELECT d.department_name, l.city FROM departments d JOIN locations l ON d.location_id = l.location_id;",
        "SQLResult":"""department_name          city
        
                          Executive            New York
                          HR                   São Paulo
                          IT                   Berlin
                          Marketing            Tokyo
                          Sales                Lagos
    
                              
                    """,
        "Answer": """department_name          city
        
                          Executive            New York
                          HR                   São Paulo
                          IT                   Berlin
                          Marketing            Tokyo
                          Sales                Lagos
    
                              
                    """
    },
    
    
    {
        "Question":"Number of departments per country",
        "SQLQuery":"SELECT c.country_name, COUNT(d.department_id) FROM departments d JOIN locations l ON d.location_id = l.location_id JOIN countries c ON l.country_id = c.country_id GROUP BY c.country_name;",
        "SQLResult":"""country_name     COUNT(d.department_id)
        
                            Brazil            1
                            Germany           1
                            Japan             1
                            Nigeria           1
                            United States     1
    
                              
                    """,
        "Answer": """country_name     COUNT(d.department_id)
        
                            Brazil            1
                            Germany           1
                            Japan             1
                            Nigeria           1
                            United States     1
    
                              
                    """,
    },
    
    
    
    
    {
        "Question":"Employees working in Germany.",
        "SQLQuery":"SELECT e.first_name, e.last_name FROM employees e JOIN departments d ON e.department_id = d.department_id JOIN locations l ON d.location_id = l.location_id WHERE l.country_id = 'DE';",
        "SQLResult":"""first_name,        last_name
        
                            Jennifer       arcia
                            Michael        Williams
                            Robert         Miller
    
                              
                    """,
        "Answer": """first_name,        last_name
        
                            Jennifer       arcia
                            Michael        Williams
                            Robert         Miller
    
                              
                    """,
    },
    
    
    
    {
        "Question":"Job titles with min and max salary",
        "SQLQuery":"SELECT job_title, min_salary, max_salary FROM jobs;",
        "SQLResult":"""job_title,                    min_salary,                     max_salary
        
                       Sales Representative            3000.00                        8000.00
                       Programmer                      4000.00                        10000.00
                       President                       20000.00                       40000.00
                       Marketing Manager               9000.00                        15000.00
                       Administration Vice President   15000.00                       30000.00
    
                              
                    """,
        "Answer": """job_title,                    min_salary,                     max_salary
        
                       Sales Representative            3000.00                        8000.00
                       Programmer                      4000.00                        10000.00
                       President                       20000.00                       40000.00
                       Marketing Manager               9000.00                        15000.00
                       Administration Vice President   15000.00                       30000.00
    
                              
                    """,
    },
    
    
    {
        "Question":"Employee with highest salary",
        "SQLQuery":"SELECT job_title, min_salary, max_salary FROM jobs;",
        "SQLResult":"""first_name          last_name           salary
        
                       John                Smith               25000.00
                         
                    """,
        "Answer": """first_name          last_name           salary
        
                       John                Smith               25000.00
                         
                    """
    },
    
    
    {
        "Question":"Show average salary by department",
        "SQLQuery":"SELECT d.department_name, AVG(e.salary) FROM departments d JOIN employees e ON d.department_id = e.department_id GROUP BY d.department_name;",
        "SQLResult":"""department_name          AVG(e.salary)
        
                       Executive                21500.000000
                       IT                       7500.000000
                       Marketing                12000.000000
                       Sales                    4250.000000
                         
                    """,
        "Answer": """department_name          AVG(e.salary)
        
                       Executive                21500.000000
                       IT                       7500.000000
                       Marketing                12000.000000
                       Sales                    4250.000000
                         
                    """
    },
    
    
    {
        "Question":"Job titles with avg salary > 9000.",
        "SQLQuery":"SELECT j.job_title, AVG(e.salary) AS avg_salary FROM employees e JOIN jobs j ON e.job_id = j.job_id GROUP BY j.job_title HAVING avg_salary > 9000;",
        "SQLResult":"""job_title                        avg_salary
        
                       Administration Vice President    18000.000000
                       Marketing Manager                12000.000000
                       President                        25000.000000
                         
                    """,
        "Answer": """job_title                        avg_salary
        
                       Administration Vice President    18000.000000
                       Marketing Manager                12000.000000
                       President                        25000.000000
                         
                    """
    },
    
    
    
    {
        "Question":"List of Employees hired before 2014",
        "SQLQuery":"SELECT * FROM employees WHERE hire_date < '2014-01-01';",
        "SQLResult":"""employee_id     first_name      last_name      email                            phone_number     hire_date    job_id    salary         manager_id, department_id
                       
                        1              John            Smith          john.smith@company.com           555-1001         2010-06-01    1        25000.00                      1
                        2              Sarah           Johnson        sarah.johnson@company.com        555-1002         2011-07-15    2        18000.00            1         1
                        3              Michael         Williams       michael.williams@company.com     555-1003         2012-03-10    3        8000.00             2         2
                        4              Emily           Brown          emily.brown@company.com          555-1004         2013-09-22    4        12000.00            2         3
                      
                    """,
        "Answer": """employee_id     first_name      last_name      email                            phone_number     hire_date    job_id    salary         manager_id, department_id
                        
                        1              John            Smith          john.smith@company.com           555-1001         2010-06-01    1        25000.00                      1
                        2              Sarah           Johnson        sarah.johnson@company.com        555-1002         2011-07-15    2        18000.00            1         1
                        3              Michael         Williams       michael.williams@company.com     555-1003         2012-03-10    3        8000.00             2         2
                        4              Emily           Brown          emily.brown@company.com          555-1004         2013-09-22    4        12000.00            2         3
                     
                    """
    },
    
    
    
    
    {
        "Question":"Total salary paid in each region.",
        "SQLQuery":"SELECT r.region_name, SUM(e.salary) FROM employees e JOIN departments d ON e.department_id = d.department_id JOIN locations l ON d.location_id = l.location_id JOIN countries c ON l.country_id = c.country_id JOIN regions r ON c.region_id = r.region_id GROUP BY r.region_name;",
        "SQLResult":"""region_name                      SUM(e.salary)
        
                       Asia                             12000.00
                       Middle East and Africa           17000.00
                       Europe                           22500.00
                       Americas                         43000.00
                         
                    """,
        "Answer": """region_name                      SUM(e.salary)
        
                       Asia                             12000.00
                       Middle East and Africa           17000.00
                       Europe                           22500.00
                       Americas                         43000.00
                         
                    """
    },
    
    
    
    
    {
        "Question":"Top 3 departments by number of employees",
        "SQLQuery":"SELECT department_id, COUNT(*) AS count FROM employees GROUP BY department_id ORDER BY count DESC LIMIT 3;",
        "SQLResult":"""department_id                count
                            
                            1                         2
                            2                         3
                            4                         4
                         
                    """,
        "Answer": """department_id                count
                            
                            1                         2
                            2                         3
                            4                         4
                         
                    """
    },
    
    
    
    {
        "Question":"Top 3 departments by number of employees",
        "SQLQuery":"SELECT department_id, COUNT(*) AS count FROM employees GROUP BY department_id ORDER BY count DESC LIMIT 3;",
        "SQLResult":"""job_title                        AVG(e.salary)
                       
                       Administration Vice President    18000.000000
                       Marketing Manager                12000.000000
                       President                        25000.000000
                       Programmer                       7500.000000
                       Sales Representative             4250.000000
                         
                    """,
        "Answer": """job_title                        AVG(e.salary)
                       
                       Administration Vice President    18000.000000
                       Marketing Manager                12000.000000
                       President                        25000.000000
                       Programmer                       7500.000000
                       Sales Representative             4250.000000
                         
                    """
    },
    
    
    {
        "Question":"List of all employees and their email addresses.",
        "SQLQuery":"SELECT first_name, last_name, email FROM employees;",
        "SQLResult":"""first_name       last_name       email
                       
                       David            Jones           david.jones@company.com
                       Emily            Brown           emily.brown@company.com
                       Jennifer         Garcia          jennifer.garcia@company.com
                       John             Smith           john.smith@company.com
                       Lisa             Davis           lisa.davis@company.com
                       Michael          Williams        michael.williams@company.com
                       Patricia         Martinez        patricia.martinez@company.com
                       Robert           Miller          robert.miller@company.com
                       Sarah            Johnson         sarah.johnson@company.com
                       Thomas           Rodriguez       thomas.rodriguez@company.com
                         
                    """,
        "Answer": """first_name       last_name       email
                       
                       David            Jones           david.jones@company.com
                       Emily            Brown           emily.brown@company.com
                       Jennifer         Garcia          jennifer.garcia@company.com
                       John             Smith           john.smith@company.com
                       Lisa             Davis           lisa.davis@company.com
                       Michael          Williams        michael.williams@company.com
                       Patricia         Martinez        patricia.martinez@company.com
                       Robert           Miller          robert.miller@company.com
                       Sarah            Johnson         sarah.johnson@company.com
                       Thomas           Rodriguez       thomas.rodriguez@company.com
                         
                    """
    },
    
    
    
    {
        "Question":"List employee name and job title for salary > 8000.",
        "SQLQuery":"SELECT e.first_name, e.last_name, j.job_title FROM employees e JOIN jobs j ON e.job_id = j.job_id WHERE e.salary > 8000;",
        "SQLResult":"""first_name       last_name      job_title
                       
                       John              Smith          President
                       Sarah             Johnson        Administration Vice President
                       Emily             Brown          Marketing Manager
                         
                    """,
        "Answer": """first_name       last_name      job_title
                       
                       John              Smith          President
                       Sarah             Johnson        Administration Vice President
                       Emily             Brown          Marketing Manager
                         
                    """
    },
    
    
    
    {
        "Question":"All regions and number of employees in each",
        "SQLQuery":"SELECT r.region_name, COUNT(e.employee_id) FROM employees e JOIN departments d ON e.department_id = d.department_id JOIN locations l ON d.location_id = l.location_id JOIN countries c ON l.country_id = c.country_id JOIN regions r ON c.region_id = r.region_id GROUP BY r.region_name;",
        "SQLResult":"""region_name            COUNT(e.employee_id)
                       
                       Americas                    2
                       Asia                        1
                       Europe                      3
                       Middle East and Africa      4
                         
                    """,
        "Answer": """region_name            COUNT(e.employee_id)
                       
                       Americas                    2
                       Asia                        1
                       Europe                      3
                       Middle East and Africa      4
                         
                    """
    },
    
    
    
    {
        "Question":"Departments with average salary > 10,000",
        "SQLQuery":"SELECT d.department_name, AVG(e.salary) AS avg_salary FROM employees e JOIN departments d ON e.department_id = d.department_id GROUP BY d.department_name HAVING avg_salary > 10000;",
        "SQLResult":"""department_name       avg_salary
                       
                       Executive             21500.000000
                       Marketing             12000.000000
                         
                    """,
        "Answer": """department_name         avg_salary
                       
                       Executive             21500.000000
                       Marketing             12000.000000
                  """
    },
    
    
    
    
    {
        "Question":"Names of countries where 'Sales' departments are located.",
        "SQLQuery":"SELECT DISTINCT c.country_name FROM departments d JOIN locations l ON d.location_id = l.location_id JOIN countries c ON l.country_id = c.country_id WHERE d.department_name = 'Sales';",
        "SQLResult":"""country_name        
                       
                        Nigeria

                        
                         
                    """,
        "Answer": """country_name        
                       
                        Nigeria
                  """
    },
    
    
    
    {
        "Question":"Find employees who are managers.",
        "SQLQuery":"SELECT DISTINCT m.first_name, m.last_name FROM employees e JOIN employees m ON e.manager_id = m.employee_id;",
        "SQLResult":"""first_name  last_name        
                       
                        John        Smith
                        Sarah       Johnson
                        Michael     Williams
                        Emily       Brown
                        David       Jones

                        
                         
                    """,
        "Answer": """first_name  last_name        
                       
                        John        Smith
                        Sarah       Johnson
                        Michael     Williams
                        Emily       Brown
                        David       Jones
                  """
    },
    
    
    
    
    {
        "Question":"Find employees who do not manage anyone",
        "SQLQuery":"SELECT first_name, last_name FROM employees WHERE employee_id NOT IN (SELECT DISTINCT manager_id FROM employees WHERE manager_id IS NOT NULL);",
        "SQLResult":"""first_name  last_name        
                       
                        Jennifer   Garcia
                        Robert     Miller
                        Lisa       Davis
                        Thomas     Rodriguez
                        Patricia   Martinez

                        
                         
                    """,
        "Answer": """first_name  last_name        
                       
                        Jennifer   Garcia
                        Robert     Miller
                        Lisa       Davis
                        Thomas     Rodriguez
                        Patricia   Martinez
                  """
    },
    
    
    
    {
        "Question":"List dependents whose parents are in 'Marketing'.",
        "SQLQuery":"SELECT d.first_name, d.last_name FROM dependents d JOIN employees e ON d.employee_id = e.employee_id JOIN departments dep ON e.department_id = dep.department_id WHERE dep.department_name = 'Marketing';",
        "SQLResult":"""first_name  last_name        
                       
                        Sophia      Brown
              
                    """,
        "Answer": """first_name  last_name        
                       
                    Sophia       Brown
                  """
    },
    
    
    
    
    
    {
        "Question":"List job titles available in the company",
        "SQLQuery":"SELECT DISTINCT job_title FROM jobs;",
        "SQLResult":"""job_title       
                       
                        President
                        Administration Vice President
                        Programmer
                        Marketing Manager
                        Sales Representative
              
                    """,
        "Answer": """  job_title       
                       
                        President
                        Administration Vice President
                        Programmer
                        Marketing Manager
                        Sales Representative
                  """
    },
    
    
    
    
    {
        "Question":"Get list of departments without any employees.",
        "SQLQuery":"SELECT department_name FROM departments WHERE department_id NOT IN (SELECT DISTINCT department_id FROM employees);",
        "SQLResult":"""department_name       
                       
                        HR
              
                    """,
        "Answer": """  department_name       
                       
                        HR
                  """
    },
    
    
    {
        "Question":"how job_id and total salary per job",
        "SQLQuery":"SELECT job_id, SUM(salary) FROM employees GROUP BY job_id;",
        "SQLResult":"""job_id     SUM(salary)       
                       
                        1           25000.00
                        2           18000.00
                        3           22500.00
                        4           12000.00
                        5           17000.00
              
                    """,
        "Answer": """  job_id     SUM(salary)       
                       
                        1           25000.00
                        2           18000.00
                        3           22500.00
                        4           12000.00
                        5           17000.00
                  """
    },
    
    
    
    {
        "Question":"List the total number of dependents per relationship type.",
        "SQLQuery":"SELECT relationship, COUNT(*) FROM dependents GROUP BY relationship;",
        "SQLResult":"""relationship  COUNT(*)       
                       
                         Child          8
                         Spouse         2
              
                    """,
        "Answer": """  relationship  COUNT(*)       
                       
                         Child          8
                         Spouse         2
                  """
    },
]

In [95]:
emb = OllamaEmbeddings(model="nomic-embed-text")

In [96]:
to_victorize = ["".join(str(i) for i in sample.values()) for sample in few_shorts]
to_victorize[0]

"List all employees' first and last names.SELECT first_name, last_name FROM employees;first_name, last_name\n        \n                       Jhon        Smith\n                       Sarah       Johnson\n                       Michael     Williams \n                       Emily       Brown\n                       David       Jones\n                       Jennifer    Garcia\n                       Robert      Miller\n                       Lisa        Davis\n                       Thomas      Rodriguez\n                       Patricia    Martinez\n\n                    first_name, last_name\n        \n                       Jhon        Smith\n                       Sarah       Johnson\n                       Michael     Williams \n                       Emily       Brown\n                       David       Jones\n                       Jennifer    Garcia\n                       Robert      Miller\n                       Lisa        Davis\n                       Thomas      Rodriguez\n 

In [97]:
vector_db = Chroma.from_texts(to_victorize, embedding=emb, metadatas=few_shorts)

In [98]:
sample_selector = SemanticSimilarityExampleSelector(
    vectorstore=vector_db,
    k=2,
)

In [99]:
sample_selector.select_examples({"Question":"Show all employees hired after 2015"})

[{'Answer': 'first_name  last_name    hire_date\n        \n                       Jennifer    Garcia     2015-02-18\n                       Robert      Miller     2016-04-30\n                       Lisa        Davis      2017-08-12\n                       Thomas      Rodriguez  2018-01-25\n                       Patricia    Martinez   2019-05-08\n                       \n\n                    ',
  'Question': 'Show all employees hired after 2015',
  'SQLQuery': "SELECT first_name, last_name, hire_date FROM employees WHERE hire_date > '2015-01-01';",
  'SQLResult': 'first_name  last_name  hire_date\n        \n                       Jennifer    Garcia     2015-02-18\n                       Robert      Miller     2016-04-30\n                       Lisa        Davis      2017-08-12\n                       Thomas      Rodriguez  2018-01-25\n                       Patricia    Martinez   2019-05-08\n                       \n\n                    '},
 {'Answer': 'first_name  last_name    hire_

In [100]:
print(_mysql_prompt)

You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of

In [101]:
prompt_templete = PromptTemplate(
    input_variables=['Question', 'SQLQuery', 'SQLResult', 'Answer'],
    template="\nQuestion : {Question}\nSQLQuery : {SQLQuery}\nSQLResult : {SQLResult}\nAnswer"
)

In [102]:
few_short_prompt = FewShotPromptTemplate(
    example_selector=sample_selector,
    example_prompt=prompt_templete,
    prefix=_mysql_prompt,
    suffix=PROMPT_SUFFIX,
    input_variables=['input', 'table_info', 'top_k', 'agent_scratchpad'],
)

In [103]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", few_short_prompt.format(input="User's input goes here", table_info="Table info here", top_k="5")),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [104]:
new_db_chain = create_sql_agent(
    llm=llm, 
    db=db,
    verbose = True,
    agent_type="openai-tools",
    use_query_checker=True,
    prompt=chat_prompt
)

In [105]:
new_chat_1 = new_db_chain('Number of departments per country')
new_chat_1



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_query` with `{'query': 'SELECT COUNT(*), `country` FROM `departments` GROUP BY `country`'}`


Error: (pymysql.err.OperationalError) (1054, "Unknown column 'country' in 'field list'")
[SQL: SELECT COUNT(*), `country` FROM `departments` GROUP BY `country`]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Invoking: `sql_db_schema` with `{'table_names': 'departments'}`



CREATE TABLE departments (
	department_id INTEGER(11) NOT NULL AUTO_INCREMENT, 
	department_name VARCHAR(30) NOT NULL, 
	location_id INTEGER(11), 
	PRIMARY KEY (department_id), 
	CONSTRAINT departments_ibfk_1 FOREIGN KEY(location_id) REFERENCES locations (location_id) ON DELETE CASCADE ON UPDATE CASCADE
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB

/*
3 rows from departments table:
department_id	department_name	location_id
1	Executive	1
2	IT	2
3	Marketing	3
*/
Invoking: `sql_db_schema` with `{'table_names': 'locations'}`



CREATE TABLE locations (
	loc

{'input': 'Number of departments per country',
 'output': 'Answer:\n\nThe number of departments per country are as follows: BR: 1, DE: 1, JP: 1, NG: 1, US: 1.'}

In [106]:
new_chat_2 = new_db_chain('List of all employees and their email addresses.')
new_chat_2



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_query` with `{'query': 'SELECT `first_name`, `last_name`, `email` FROM `employees` LIMIT 5;'}`


[('John', 'Smith', 'john.smith@company.com'), ('Sarah', 'Johnson', 'sarah.johnson@company.com'), ('Michael', 'Williams', 'michael.williams@company.com'), ('Emily', 'Brown', 'emily.brown@company.com'), ('David', 'Jones', 'david.jones@company.com')]Here are some employees and their email addresses: John Smith (john.smith@company.com), Sarah Johnson (sarah.johnson@company.com), Michael Williams (michael.williams@company.com), Emily Brown (emily.brown@company.com), David Jones (david.jones@company.com).

> Finished chain.


{'input': 'List of all employees and their email addresses.',
 'output': 'Here are some employees and their email addresses: John Smith (john.smith@company.com), Sarah Johnson (sarah.johnson@company.com), Michael Williams (michael.williams@company.com), Emily Brown (emily.brown@company.com), David Jones (david.jones@company.com).'}